# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [5]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

ValueError: GitHub 저장소 URL을 입력해야 합니다.

In [ ]:
!python --version
!nvidia-smi
!pip install -r ./requirements.txt

Python 3.12.13
Mon Jun  1 14:21:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
저장됨: /content/week14-team-05-gpt-lab/data/ratings_train.txt
다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
저장됨: /content/week14-team-05-gpt-lab/data/ratings_test.txt
사전 학습 train 텍스트: /content/week14-team-05-gpt-lab/data/nsmc_lm_train.txt (1,379,486자)
사전 학습 val 텍스트: /content/week14-team-05-gpt-lab/data/nsmc_lm_val.txt (120,560자)
감성 분류 train: /content/week14-team-05-gpt-lab/data/nsmc_sentiment_train.jsonl (137,996개)
감성 분류 val: /content/week14-team-05-gpt-lab/data/nsmc_sentiment_val.jsonl (11,999개)
감성 분류 test: /content/week14-team-05-gpt-lab/data/nsmc_sentiment_test.jsonl (49,997개)
LM train exists: True /content/week14-team-05-gpt-lab/data/nsmc_lm_train.txt
LM val exists: True /content/week14-team-05-gpt-lab/data/nsmc_lm_val.txt


In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

train chars: 1379486
val chars: 120560
개재미없다. 감독의 연출력의 한계
이제서야 보게된 대 명작 연출미가 정말 훌륭하다!!!!!!!!
소주미라클을 만들어라
귀여운 캐릭터들도 많이 나와서 보러 가야 겠어요..
블랙 코미디가 싫어요.
평점깎고싶다10글자
TV시리즈가 너무재밌어서 영화는 기대안하고 봤는데 역시....최고네요
개인적 공감이 글쎄?
시작은 니시지마 때문에 봤는데 나름 괜찮은 영화 봤다고


## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_bpe.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week14-team-05-gpt-lab
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collecting ... collected 6 items

tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [ 16%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 33%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 50%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 66%]
tests/test_bpe.py::TestBPETokenizer::test_get_special_ids PASSED         [ 83%]
tests/test_bpe.py::TestBPETrain::test_train_increases_vocab PASSED       [100%]

============================== 6 passed in 0.03s ===============================


선택한 테스트를 통과했습니다.


0

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

[2, 265, 260, 290, 268, 260, 164, 153, 274, 148, 260, 166, 143, 281, 156, 270, 37, 36, 73, 114]
이 영화는 정말 좋았다! English 123


## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_dataset.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week14-team-05-gpt-lab
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collecting ... collected 4 items

tests/test_dataset.py::TestGPTDataset::test_dataset_length PASSED        [ 25%]
tests/test_dataset.py::TestGPTDataset::test_dataset_getitem_shape PASSED [ 50%]
tests/test_dataset.py::TestCreateDataloader::test_dataloader_batch_shape PASSED [ 75%]
tests/test_dataset.py::TestInputEmbedding::test_input_embedding_shape PASSED [100%]

============================== 4 passed in 5.28s ===============================


선택한 테스트를 통과했습니다.


0

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

torch.Size([2, 32]) torch.Size([2, 32]) torch.Size([2, 32, 32])


## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_attention.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week14-team-05-gpt-lab
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collecting ... collected 2 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [ 50%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [100%]

============================== 2 passed in 2.14s ===============================


선택한 테스트를 통과했습니다.


0

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_model.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week14-team-05-gpt-lab
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collecting ... collected 7 items

tests/test_model.py::TestLayerNorm::test_layernorm_shape PASSED          [ 14%]
tests/test_model.py::TestGELU::test_gelu_shape PASSED                    [ 28%]
tests/test_model.py::TestFeedForward::test_feedforward_shape PASSED      [ 42%]
tests/test_model.py::TestTransformerBlock::test_transformer_block_shape PASSED [ 57%]
tests/test_model.py::TestGPTModel::test_gpt_forward_shape PASSED         [ 71%]
tests/test_model.py::TestGPTModel::test_gpt_forward_with_targets_returns_loss PASSED [ 85%]
tests/test_model.py::TestGenerateTextSimple::test_generate_text_simple_shape PASSED [100%]

============================== 7 pas

0

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

torch.Size([2, 16, 300])


## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_train.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week14-team-05-gpt-lab
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collecting ... collected 5 items

tests/test_train.py::TestCalcLossBatch::test_calc_loss_batch_returns_scalar PASSED [ 20%]
tests/test_train.py::TestCalcLossLoader::test_calc_loss_loader_returns_float PASSED [ 40%]
tests/test_train.py::TestCheckpoint::test_save_load_checkpoint_restores_epoch_and_step PASSED [ 60%]
tests/test_train.py::TestGenerate::test_generate_shape PASSED            [ 80%]
tests/test_train.py::TestPlotLosses::test_plot_losses_callable PASSED    [100%]

============================== 5 passed in 7.60s ===============================


선택한 테스트를 통과했습니다.


0

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

smoke loss: 5.958449363708496


## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_finetune.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week14-team-05-gpt-lab
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collecting ... collected 4 items

tests/test_finetune.py::TestMakeSentimentDataset::test_make_sentiment_dataset_splits_rows PASSED [ 25%]
tests/test_finetune.py::TestReviewSentimentDataset::test_review_sentiment_dataset_getitem PASSED [ 50%]
tests/test_finetune.py::TestGPTForSequenceClassification::test_sequence_classification_shape PASSED [ 75%]
tests/test_finetune.py::TestSentimentTrainEval::test_train_eval_functions_exist PASSED [100%]

============================== 4 passed in 1.62s ===============================


선택한 테스트를 통과했습니다.


0

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")

실행 명령: /usr/bin/python3 -m pytest tests/ -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week14-team-05-gpt-lab
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collecting ... collected 28 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [  3%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [  7%]
tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [ 10%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 14%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 17%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 21%]
tests/test_bpe.py::TestBPETokenizer::test_get_special_ids PASSED         [ 25%]
tests/test_bpe.py::TestBPETrain::test_train

0

# Python, PyTorch, CUDA, 주요 패키지 버전을 기록

In [11]:
import platform
import random
import sys

import matplotlib
import numpy as np
import torch

print("python:", sys.version)
print("python_major_minor:", f"{sys.version_info.major}.{sys.version_info.minor}")
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_version:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
python_major_minor: 3.12
platform: Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.11.0+cu128
cuda_available: True
cuda_version: 12.8
gpu: Tesla T4
numpy: 2.0.2
matplotlib: 3.10.0


## A0_basic 실험

Colab GPU에서 Basic 제출 기준 baseline을 실행합니다. 공유 BPE tokenizer `artifacts/tokenizers/nsmc_bpe_vocab3000_full.json`을 로드하고, W&B가 로그인되어 있으면 online으로, 아니면 offline으로 기록합니다.

In [2]:
import sys

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("현재 커널은 Colab이 아니므로 Drive mount를 건너뜁니다.")


Mounted at /content/drive


In [3]:
from pathlib import Path

output_root = Path("/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain")
output_root.mkdir(parents=True, exist_ok=True)
print(output_root.exists(), output_root)


True /content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain


In [5]:
%cd /content

!git clone https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git

%cd /content/week14-team-05-gpt-lab
!git checkout jaehwan-experiment-a-colab
!git pull

!python experiments/scripts/run_a_pretrain_stability.py \
  --experiment A0_basic \
  --train-char-limit 1500000 \
  --vocab-size 3000 \
  --output-root /content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain \
  --device cuda \
  --wandb \
  --wandb-mode offline


/content
Cloning into 'week14-team-05-gpt-lab'...
remote: Enumerating objects: 565, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 565 (delta 45), reused 75 (delta 25), pack-reused 393 (from 1)
Receiving objects: 100% (565/565), 246.36 KiB | 8.80 MiB/s, done.
Resolving deltas: 100% (348/348), done.
/content/week14-team-05-gpt-lab
Branch 'jaehwan-experiment-a-colab' set up to track remote branch 'jaehwan-experiment-a-colab' from 'origin'.
Switched to a new branch 'jaehwan-experiment-a-colab'
Already up to date.
$ /usr/bin/python3 download_data.py
다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
저장됨: /content/week14-team-05-gpt-lab/data/ratings_train.txt
다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
저장됨: /content/week14-team-05-gpt-lab/data/ratings_test.txt
사전 학습 train 텍스트: /content/week14-team-05-gpt-lab/data/nsmc_lm_train.txt (1,379,486자)
사전 학습 val 텍스트: /conte

## A1: warmup + cosine decay 재실험

기존 A1(20260602)은 `context_length=64` 기준으로 실행된 것으로 보여 `context_length=128`인 A0_basic과 직접 비교하기 어렵습니다. 이 재실험은 A0_basic과 같은 Basic 기준에서 scheduler 효과만 다시 보기 위해 `context_length=128`, `warmup_steps=50`, `min_lr_ratio=0.1`로 실행합니다. 결과는 기존 산출물과 섞이지 않도록 `A1_20260603_JAEHWAN` 경로에 저장합니다.

In [10]:
%cd /content

!git clone https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git

%cd /content/week14-team-05-gpt-lab
!git checkout jaehwan-experiment-a-colab
!git pull

/content
fatal: destination path 'week14-team-05-gpt-lab' already exists and is not an empty directory.
/content/week14-team-05-gpt-lab
Already on 'jaehwan-experiment-a-colab'
Your branch is up to date with 'origin/jaehwan-experiment-a-colab'.
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 31 (delta 21), reused 23 (delta 13), pack-reused 0 (from 0)
Unpacking objects: 100% (31/31), 39.52 KiB | 1.72 MiB/s, done.
From https://github.com/Jungle-12-303/week14-team-05-gpt-lab
   88a39a0..2d87bbb  jaehwan-experiment-a-colab -> origin/jaehwan-experiment-a-colab
   bab3567..82063e2  main       -> origin/main
 * [new branch]      test/yb    -> origin/test/yb
Updating 88a39a0..2d87bbb
Fast-forward
 docs/EXPERIMENT_A_JAEHWAN.md                    |   7 +-
 experiments/scripts/_pretrain_runner.py         |   6 +-
 experiments/scripts/run_a_pretrain_stability.py |   9 +-
 gpt-lab.ipynb               

In [12]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "jaehwan-experiment-a-colab"
REPO_URL = "https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git"
REPO_DIR = Path("/content/week14-team-05-gpt-lab")
OUTPUT_ROOT = Path("/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain")
RUN_DATE = "20260603"  # 기존 A1_20260602 결과와 분리

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("현재 커널은 Colab이 아니므로 Drive mount를 건너뜁니다.")

if not REPO_DIR.exists():
    run(["git", "clone", REPO_URL, str(REPO_DIR)])

run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
run(["git", "checkout", BRANCH], cwd=REPO_DIR)
run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR)

import torch

print("cuda_available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    raise RuntimeError("Colab runtime GPU가 켜져 있지 않습니다. Runtime > Change runtime type > GPU로 바꾼 뒤 다시 실행하세요.")

tokenizer_path = REPO_DIR / "artifacts/tokenizers/nsmc_bpe_vocab3000_full.json"
if not tokenizer_path.exists():
    raise FileNotFoundError(f"Shared tokenizer not found: {tokenizer_path}")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "offline"
print("wandb_mode:", wandb_mode)

run([
    sys.executable,
    "experiments/scripts/run_a_pretrain_stability.py",
    "--experiment", "A1",
    "--date", RUN_DATE,
    "--train-char-limit", "1500000",
    "--vocab-size", "3000",
    "--output-root", str(OUTPUT_ROOT),
    "--device", "cuda",
    "--wandb",
    "--wandb-mode", wandb_mode,
], cwd=REPO_DIR)

summary_path = OUTPUT_ROOT / f"A1_{RUN_DATE}_JAEHWAN" / "summary.json"
print("summary exists:", summary_path.exists(), summary_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin jaehwan-experiment-a-colab
$ git checkout jaehwan-experiment-a-colab
$ git pull --ff-only origin jaehwan-experiment-a-colab
$ /usr/bin/python3 -m pip install -r requirements.txt
cuda_available: True
gpu: Tesla T4
wandb_mode: offline
$ /usr/bin/python3 experiments/scripts/run_a_pretrain_stability.py --experiment A1 --date 20260603 --train-char-limit 1500000 --vocab-size 3000 --output-root /content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain --device cuda --wandb --wandb-mode offline
summary exists: True /content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A1_20260603_JAEHWAN/summary.json


In [13]:
from pathlib import Path
import json

RUN_DATE = "20260603"
summary_path = Path(f"/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A1_{RUN_DATE}_JAEHWAN/summary.json")
print(json.dumps(json.loads(summary_path.read_text()), ensure_ascii=False, indent=2))


{
  "suite": "A_pretrain_stability",
  "experiment_id": "A1",
  "group": null,
  "change": "warmup + cosine decay, Basic context, min lr floor",
  "value": null,
  "output_dir": "/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A1_20260603_JAEHWAN",
  "metrics_path": "/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A1_20260603_JAEHWAN/metrics/A1_20260603_metrics.jsonl",
  "log_path": "/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A1_20260603_JAEHWAN/logs/A1_20260603.out",
  "wandb_dir": "/content/drive/MyDrive/gpt-lab/wandb",
  "tokenizer_path": "/content/week14-team-05-gpt-lab/artifacts/tokenizers/nsmc_bpe_vocab3000_full.json",
  "best_checkpoint_path": "/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A1_20260603_JAEHWAN/checkpoints/A1_20260603_step1574_best.pt",
  "latest_checkpoint_paths": [
    "/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A1_20260603_JAEHWAN/checkpoints/A1_20260603_step1400_latest.pt",
    "/content/drive/

In [14]:
from pathlib import Path

RUN_DATE = "20260603"
metrics_path = Path(f"/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A1_{RUN_DATE}_JAEHWAN/metrics/A1_{RUN_DATE}_metrics.jsonl")
for line in metrics_path.read_text().strip().splitlines()[-10:]:
    print(line)


{"event": "train_step", "timestamp": "2026-06-02T17:02:06", "experiment_id": "A1", "epoch": 2, "global_step": 1440, "train_loss": 7.270328044891357, "learning_rate": 3.504213812690539e-05}
{"event": "train_step", "timestamp": "2026-06-02T17:02:07", "experiment_id": "A1", "epoch": 2, "global_step": 1460, "train_loss": 7.266793251037598, "learning_rate": 3.364607683398076e-05}
{"event": "train_step", "timestamp": "2026-06-02T17:02:07", "experiment_id": "A1", "epoch": 2, "global_step": 1480, "train_loss": 7.234652042388916, "learning_rate": 3.2473255191396307e-05}
{"event": "train_step", "timestamp": "2026-06-02T17:02:07", "experiment_id": "A1", "epoch": 2, "global_step": 1500, "train_loss": 7.2870683670043945, "learning_rate": 3.1525666442194135e-05}
{"event": "eval", "timestamp": "2026-06-02T17:02:07", "experiment_id": "A1", "epoch": 2, "global_step": 1500, "train_loss": 7.252337408065796, "val_loss": 7.251577520370484, "best_val_loss": 7.251577520370484, "is_best": true, "checkpoint_pa

## A2: gradient clipping 실험

A0_basic과 같은 Basic 기준에서 gradient clipping만 적용합니다. `run_a_pretrain_stability.py`의 A2 설정은 `context_length=128`, `grad_clip_norm=1.0`이며, A0_basic best validation loss 6.7148과 비교해 clipping이 학습 안정성과 validation loss를 개선하는지 확인합니다. 결과는 `A2_20260603_JAEHWAN` 경로에 저장합니다.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "jaehwan-experiment-a-colab"
REPO_URL = "https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git"
REPO_DIR = Path("/content/week14-team-05-gpt-lab")
OUTPUT_ROOT = Path("/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain")
RUN_DATE = "20260603"

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("현재 커널은 Colab이 아니므로 Drive mount를 건너뜁니다.")

if not REPO_DIR.exists():
    run(["git", "clone", REPO_URL, str(REPO_DIR)])

run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
run(["git", "checkout", BRANCH], cwd=REPO_DIR)
run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR)

import torch

print("cuda_available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    raise RuntimeError("Colab runtime GPU가 켜져 있지 않습니다. Runtime > Change runtime type > GPU로 바꾼 뒤 다시 실행하세요.")

tokenizer_path = REPO_DIR / "artifacts/tokenizers/nsmc_bpe_vocab3000_full.json"
if not tokenizer_path.exists():
    raise FileNotFoundError(f"Shared tokenizer not found: {tokenizer_path}")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "offline"
print("wandb_mode:", wandb_mode)

run([
    sys.executable,
    "experiments/scripts/run_a_pretrain_stability.py",
    "--experiment", "A2",
    "--date", RUN_DATE,
    "--train-char-limit", "1500000",
    "--vocab-size", "3000",
    "--output-root", str(OUTPUT_ROOT),
    "--device", "cuda",
    "--wandb",
    "--wandb-mode", wandb_mode,
], cwd=REPO_DIR)

summary_path = OUTPUT_ROOT / f"A2_{RUN_DATE}_JAEHWAN" / "summary.json"
print("summary exists:", summary_path.exists(), summary_path)


In [ ]:
from pathlib import Path
import json

RUN_DATE = "20260603"
summary_path = Path(f"/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A2_{RUN_DATE}_JAEHWAN/summary.json")
print(json.dumps(json.loads(summary_path.read_text()), ensure_ascii=False, indent=2))


In [ ]:
from pathlib import Path

RUN_DATE = "20260603"
metrics_path = Path(f"/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain/A2_{RUN_DATE}_JAEHWAN/metrics/A2_{RUN_DATE}_metrics.jsonl")
for line in metrics_path.read_text().strip().splitlines()[-10:]:
    print(line)


## A2-A4 자동 실행 및 리포트 반영

아래 셀 하나만 실행하면 A2, A3, A4를 순서대로 실행하고 `docs/EXPERIMENT_A_JAEHWAN.md`와 `REPORT.md`에 결과를 반영합니다. Colab 연결 유지를 위한 keep-alive JavaScript도 함께 등록합니다.

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

from IPython.display import Javascript, display

BRANCH = "jaehwan-experiment-a-colab"
REPO_URL = "https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git"
REPO_DIR = Path("/content/week14-team-05-gpt-lab")
OUTPUT_ROOT = Path("/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain")
OWNER = "JAEHWAN"
RUN_DATE = datetime.now().strftime("%Y%m%d")
EXPERIMENT_IDS = ["A2", "A3", "A4"]
TRAIN_CHAR_LIMIT = "1500000"
VOCAB_SIZE = "3000"
BASELINE_BEST_VAL_LOSS = 6.7148

KEEP_ALIVE_JS = """
(() => {
  if (window.gptLabKeepAliveInterval) {
    clearInterval(window.gptLabKeepAliveInterval);
  }
  const clickConnect = () => {
    const selectors = [
      'colab-connect-button',
      '#connect',
      'paper-button#connect',
      'colab-toolbar-button#connect'
    ];
    for (const selector of selectors) {
      const el = document.querySelector(selector);
      if (el) {
        const button = el.shadowRoot?.querySelector('#connect') || el;
        if (button && !button.disabled) button.click();
        break;
      }
    }
    console.log('[gpt-lab] keep-alive tick', new Date().toISOString());
  };
  window.gptLabKeepAliveInterval = setInterval(clickConnect, 60000);
  console.log('[gpt-lab] Colab keep-alive registered');
})();
"""
display(Javascript(KEEP_ALIVE_JS))


def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True, env=env)


def mount_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ModuleNotFoundError:
        print("현재 커널은 Colab이 아니므로 Drive mount를 건너뜁니다.")


def git_tree_dirty():
    result = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True, capture_output=True, check=True)
    return bool(result.stdout.strip())


def prepare_repo():
    if not REPO_DIR.exists():
        run(["git", "clone", REPO_URL, str(REPO_DIR)])
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    if git_tree_dirty():
        print("local report changes detected; skipping git pull to keep generated results")
    else:
        run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
    run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR)
    sys.path.insert(0, str(REPO_DIR / "src"))


def patch_a_experiment_script():
    path = REPO_DIR / "experiments/scripts/run_a_pretrain_stability.py"
    text = path.read_text(encoding="utf-8")
    text = text.replace(
        '''        "change": "weight_decay=0.01",
        "overrides": {"weight_decay": 0.01},''',
        '''        "change": "weight_decay=0.01, Basic context",
        "overrides": {
            "context_length": 128,
            "weight_decay": 0.01,
        },''',
    )
    text = text.replace(
        '''        "change": "warmup + cosine + clipping + weight_decay",
        "overrides": {
            "scheduler": "warmup_cosine",
            "grad_clip_norm": 1.0,
            "weight_decay": 0.01,
        },''',
        '''        "change": "warmup + cosine + clipping + weight_decay, Basic context",
        "overrides": {
            "context_length": 128,
            "scheduler": "warmup_cosine",
            "warmup_steps": 50,
            "min_lr_ratio": 0.1,
            "grad_clip_norm": 1.0,
            "weight_decay": 0.01,
        },''',
    )
    path.write_text(text, encoding="utf-8")
    print("A3/A4 Basic context patch checked:", path)


def validate_runtime():
    import torch

    print("cuda_available:", torch.cuda.is_available())
    print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
    if not torch.cuda.is_available():
        raise RuntimeError("Colab runtime GPU가 켜져 있지 않습니다. Runtime > Change runtime type > GPU로 바꾼 뒤 다시 실행하세요.")
    tokenizer_path = REPO_DIR / "artifacts/tokenizers/nsmc_bpe_vocab3000_full.json"
    if not tokenizer_path.exists():
        raise FileNotFoundError(f"Shared tokenizer not found: {tokenizer_path}")


def read_jsonl(path):
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            records.append(json.loads(line))
    return records


def fmt(value, digits=4):
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def basename(path):
    return Path(path).name if path else ""


def get_baseline_loss():
    candidates = sorted(OUTPUT_ROOT.glob(f"A0_basic_*_{OWNER}/summary.json"))
    for path in reversed(candidates):
        try:
            value = json.loads(path.read_text(encoding="utf-8")).get("best_val_loss")
        except Exception:
            continue
        if value is not None:
            return float(value)
    return BASELINE_BEST_VAL_LOSS


def summarize_experiment(exp_id):
    summary_path = OUTPUT_ROOT / f"{exp_id}_{RUN_DATE}_{OWNER}" / "summary.json"
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    metrics_path = Path(summary["metrics_path"])
    metrics = read_jsonl(metrics_path)
    val_records = [r for r in metrics if r.get("event") in {"eval", "epoch_end"} and r.get("val_loss") is not None]
    final_record = val_records[-1] if val_records else (metrics[-1] if metrics else {})
    best_record = None
    best_val = summary.get("best_val_loss")
    if best_val is not None:
        best_record = min(
            val_records,
            key=lambda r: abs(float(r.get("val_loss", 1e9)) - float(best_val)),
            default=None,
        )
    best_step = (best_record or {}).get("global_step") or summary.get("final_global_step")
    final_train = final_record.get("train_loss")
    baseline = get_baseline_loss()
    delta = None if best_val is None else float(best_val) - baseline
    if delta is None:
        conclusion = "결과 확인 필요"
    elif delta < 0:
        conclusion = f"A0_basic 대비 {abs(delta):.4f} 개선"
    elif delta > 0:
        conclusion = f"A0_basic 대비 {delta:.4f} 악화"
    else:
        conclusion = "A0_basic과 동일"
    return {
        "summary": summary,
        "metrics": metrics,
        "best_record": best_record or {},
        "final_record": final_record,
        "best_step": best_step,
        "best_val": best_val,
        "final_train": final_train,
        "elapsed_min": summary.get("elapsed_min"),
        "best_checkpoint": summary.get("best_checkpoint_path"),
        "metrics_path": summary.get("metrics_path"),
        "change": summary.get("change") or exp_id,
        "conclusion": conclusion,
    }


def replace_section_row(markdown, start_heading, end_heading, exp_id, replacement):
    start = markdown.index(start_heading)
    end = markdown.index(end_heading, start)
    section = markdown[start:end]
    lines = section.splitlines()
    out = []
    replaced = False
    for line in lines:
        if line.startswith(f"| {exp_id} |"):
            if not replaced:
                out.extend(replacement.splitlines())
                replaced = True
            continue
        out.append(line)
    if not replaced:
        out.append(replacement)
    return markdown[:start] + "\n".join(out) + "\n\n" + markdown[end:]


def update_between_markers(markdown, start_marker, end_marker, content, insert_before):
    block = f"{start_marker}\n{content.rstrip()}\n{end_marker}"
    if start_marker in markdown and end_marker in markdown:
        pattern = re.compile(re.escape(start_marker) + r".*?" + re.escape(end_marker), re.S)
        return pattern.sub(block, markdown)
    insert_at = markdown.index(insert_before)
    return markdown[:insert_at].rstrip() + "\n\n" + block + "\n\n" + markdown[insert_at:]


def update_a_report(results):
    path = REPO_DIR / "docs/EXPERIMENT_A_JAEHWAN.md"
    markdown = path.read_text(encoding="utf-8")
    for exp_id, result in results.items():
        result_row = (
            f"| {exp_id} | {result['change']} | {result['best_step']} | {fmt(result['best_val'])} | "
            f"{fmt(result['final_train'])} | {fmt(result['elapsed_min'], 2)} min | "
            f"`{result['best_checkpoint']}` | `{result['metrics_path']}` | {result['conclusion']} |"
        )
        markdown = replace_section_row(markdown, "## 5. 실험 결과", "## 6. Step metric 기록", exp_id, result_row)

        best = result["best_record"]
        final = result["final_record"]
        step_rows = []
        if best:
            step_rows.append(
                f"| {exp_id} | {best.get('global_step', '')} | {best.get('epoch', '')} | {fmt(best.get('train_loss'))} | "
                f"{fmt(best.get('val_loss'))} | `{basename(best.get('checkpoint_path') or result['best_checkpoint'])}` | best val loss |"
            )
        if final and final.get("global_step") != best.get("global_step"):
            step_rows.append(
                f"| {exp_id} | {final.get('global_step', '')} | {final.get('epoch', '')} | {fmt(final.get('train_loss'))} | "
                f"{fmt(final.get('val_loss'))} | `{basename(result['best_checkpoint'])}` | final eval |"
            )
        markdown = replace_section_row(
            markdown,
            "## 6. Step metric 기록",
            "## 7. 실패 또는 중단 실험",
            exp_id,
            "\n".join(step_rows) or f"| {exp_id} |  |  |  |  |  | 결과 확인 필요 |",
        )

    summary_lines = ["## A2-A4 자동 실행 요약", "", "| 실험 ID | 변경점 | best val loss | final train loss | 결론 |", "| --- | --- | ---: | ---: | --- |"]
    for exp_id, result in results.items():
        summary_lines.append(
            f"| {exp_id} | {result['change']} | {fmt(result['best_val'])} | {fmt(result['final_train'])} | {result['conclusion']} |"
        )
    markdown = update_between_markers(
        markdown,
        "<!-- AUTO_A2_A4_SUMMARY_START -->",
        "<!-- AUTO_A2_A4_SUMMARY_END -->",
        "\n".join(summary_lines),
        "## 8. 최종 결론",
    )
    path.write_text(markdown, encoding="utf-8")
    print("updated:", path)


def update_main_report(results):
    path = REPO_DIR / "REPORT.md"
    markdown = path.read_text(encoding="utf-8")
    lines = ["### 6.4 A 실험 자동 실행 결과", "", "| 실험 ID | 변경점 | best val loss | final train loss | checkpoint | 결론 |", "| --- | --- | ---: | ---: | --- | --- |"]
    for exp_id, result in results.items():
        lines.append(
            f"| {exp_id} | {result['change']} | {fmt(result['best_val'])} | {fmt(result['final_train'])} | "
            f"`{result['best_checkpoint']}` | {result['conclusion']} |"
        )
    markdown = update_between_markers(
        markdown,
        "<!-- AUTO_A_EXPERIMENT_RESULTS_START -->",
        "<!-- AUTO_A_EXPERIMENT_RESULTS_END -->",
        "\n".join(lines),
        "## 7. 미세 조정",
    )
    path.write_text(markdown, encoding="utf-8")
    print("updated:", path)


mount_drive()
prepare_repo()
patch_a_experiment_script()
validate_runtime()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "offline"
print("run_date:", RUN_DATE)
print("wandb_mode:", wandb_mode)

results = {}
for exp_id in EXPERIMENT_IDS:
    print(f"\n===== {exp_id} start =====", flush=True)
    run([
        sys.executable,
        "experiments/scripts/run_a_pretrain_stability.py",
        "--experiment", exp_id,
        "--date", RUN_DATE,
        "--train-char-limit", TRAIN_CHAR_LIMIT,
        "--vocab-size", VOCAB_SIZE,
        "--output-root", str(OUTPUT_ROOT),
        "--device", "cuda",
        "--wandb",
        "--wandb-mode", wandb_mode,
    ], cwd=REPO_DIR)
    results[exp_id] = summarize_experiment(exp_id)
    update_a_report(results)
    update_main_report(results)
    print(json.dumps(results[exp_id]["summary"], ensure_ascii=False, indent=2), flush=True)
    print(f"===== {exp_id} done =====\n", flush=True)

print("A2-A4 자동 실행 및 리포트 반영 완료")
print(json.dumps({exp_id: {"best_val_loss": r["best_val"], "final_train_loss": r["final_train"], "conclusion": r["conclusion"]} for exp_id, r in results.items()}, ensure_ascii=False, indent=2))
